# 05 — Fault Tolerance Analysis

This notebook addresses **RQ4**: *How does each service behave under fault
conditions — injected network delay, consumer restarts, and broker outage?*

We segment each time series into pre-fault, during-fault, and post-fault windows,
then measure error rates, latency degradation, and time-to-recover.

## Imports and setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('..') / 'scripts'))
from utils import SERVICES, SERVICE_LABELS, SERVICE_COLORS, load_csv, set_plot_style
from statistical_tests import _compute_test_result
from generate_charts import fig05_fault_recovery

set_plot_style()
%matplotlib inline

# Override fault window timestamps via environment variables if available.
# Fallback: use the middle third of each time series as a proxy fault window.
FAULT_START_TS = os.getenv('FAULT_START_TS', None)
FAULT_END_TS   = os.getenv('FAULT_END_TS',   None)
print('Fault window from env:', FAULT_START_TS, '→', FAULT_END_TS)
print('(If None, middle-third approximation will be used)')

## Load all fault-scenario CSVs

We load P95 latency and error rate for both services.
The benchmark was run across three scenarios; because they share the same CSV
filenames here (all data is in one time series), the window segmentation below
is what isolates each scenario's behaviour.

In [ ]:
def _segment(s: pd.Series, fault_start=None, fault_end=None):
    """Split series into pre/during/post fault windows."""
    if fault_start and fault_end:
        pre    = s[s.index <  pd.Timestamp(fault_start, tz='UTC')]
        during = s[(s.index >= pd.Timestamp(fault_start, tz='UTC')) &
                   (s.index <= pd.Timestamp(fault_end,   tz='UTC'))]
        post   = s[s.index >  pd.Timestamp(fault_end, tz='UTC')]
    else:
        n = len(s)
        pre    = s.iloc[: n // 3]
        during = s.iloc[n // 3 : 2 * n // 3]
        post   = s.iloc[2 * n // 3 :]
    return pre, during, post

data = {}
for svc in SERVICES:
    try:
        data[f'p95_{svc}']   = load_csv(f'latency_p95_{svc}.csv')['value_ms']
        data[f'err_{svc}']   = load_csv(f'error_rate_{svc}.csv')['rate']
    except FileNotFoundError as e:
        print(f'Warning: {e}')

print('Loaded series:', list(data.keys()))

## Network delay: P95 baseline vs. fault window

With `FAULT_DELAY_MS=200` configured on both services, the P95 should rise by
approximately 200 ms.  A smaller increase indicates that the service is amortising
the delay across concurrent requests (e.g. through pipelining).

In [ ]:
for svc in SERVICES:
    key = f'p95_{svc}'
    if key not in data:
        continue
    pre, during, post = _segment(data[key], FAULT_START_TS, FAULT_END_TS)
    print(f'{SERVICE_LABELS[svc]}')
    print(f'  Pre-fault  P95 mean : {pre.mean():.2f} ms  (n={len(pre)})')
    print(f'  During     P95 mean : {during.mean():.2f} ms  (n={len(during)})')
    print(f'  Post-fault P95 mean : {post.mean():.2f} ms  (n={len(post)})')
    if len(pre) > 0 and len(during) > 0:
        r = _compute_test_result(
            pre.dropna().to_numpy(), during.dropna().to_numpy(),
            f'Pre vs During ({SERVICE_LABELS[svc]})', 'ms', n_bootstrap=1000,
        )
        print(f'  Mann-Whitney: p={r.p_value:.4f}  significant={r.significant}')
    print()

## Consumer restart: time-to-recover

Recovery time is defined as the number of seconds after the fault window ends
before the service's P95 returns to within ±10% of the pre-fault baseline.

In [ ]:
for svc in SERVICES:
    key = f'p95_{svc}'
    if key not in data:
        continue
    pre, during, post = _segment(data[key], FAULT_START_TS, FAULT_END_TS)
    baseline = pre.mean()
    threshold = baseline * 1.10

    if len(post) == 0 or baseline == 0:
        print(f'{SERVICE_LABELS[svc]}: insufficient data')
        continue

    recovered_mask = post <= threshold
    if recovered_mask.any():
        first_recovered = post[recovered_mask].index[0]
        fault_end_ts = post.index[0]
        recovery_s = (first_recovered - fault_end_ts).total_seconds()
        print(f'{SERVICE_LABELS[svc]}: recovered in {recovery_s:.0f}s  (threshold: {threshold:.1f} ms)')
    else:
        print(f'{SERVICE_LABELS[svc]}: did not recover within the observation window')

## Broker outage: error rate by phase

In [ ]:
for svc in SERVICES:
    key = f'err_{svc}'
    if key not in data:
        continue
    pre, during, post = _segment(data[key], FAULT_START_TS, FAULT_END_TS)
    print(f'{SERVICE_LABELS[svc]}')
    print(f'  Pre-fault  error rate : {pre.mean()*100:.2f}%')
    print(f'  During     error rate : {during.mean()*100:.2f}%')
    print(f'  Post-fault error rate : {post.mean()*100:.2f}%')
    print()

## Figure 5: fault recovery comparison

In [ ]:
fig05_fault_recovery()

## Recovery time summary table

In [ ]:
summary = [
    ['Network Delay', 'see above', 'see above'],
    ['Consumer Restart', 'see above', 'see above'],
    ['Broker Outage', 'see above', 'see above'],
]
pd.DataFrame(summary, columns=['Scenario', '.NET recovery (s)', 'Go recovery (s)'])

## Interpretation

**Fill in after running with real data.**

Template:

> **Network delay**: both services showed a ~200 ms P95 increase as expected.
> The Go service recovered to baseline X s faster after the delay was removed.
>
> **Consumer restart**: the .NET service took A s to re-establish consumer group
> membership vs B s for Go.  The difference reflects Confluent.Kafka's session
> timeout configuration vs kafka-go's lighter reconnect loop.
>
> **Broker outage**: error rates spiked to X% (.NET) and Y% (Go) during the
> outage window.  Both services successfully reconnected after Kafka restarted,
> though Go returned to baseline Z s sooner.
>
> **Dissertation answer (RQ4):** Go demonstrated [marginally/significantly]
> faster fault recovery across all three scenarios, primarily due to its lighter
> connection management overhead.